In [1]:
import sys
import os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
# Import local utilities
# from utils import DatasetProcessor, CellVisualizer, GeneralStats
from utils.dataset_processor import DatasetProcessor
from utils.cell_visualizer import CellVisualizer
from utils.general_stats import GeneralStats

# Import your helper functions
from helper_functions.data_loader import DataLoader  # Import the DataLoader class
from helper_functions.data_analyzer import DataAnalyzer
from helper_functions.data_aligner import DataAligner
from helper_functions.data_pipeline import DataPipeline
from helper_functions.data_plotter import Plotter

ModuleNotFoundError: No module named 'utils.GLMDataUtils'

In [ ]:
# Your datasets list
datasets = [
    ("HA1-00", "2023-06-27", 'V:'),
    ("HA1-00", "2023-06-29", 'V:'),
    ("HA1-00", "2023-07-07", 'W:'), #squinty after 30k
    ("HA10-1L", "2023-03-31", 'V:'),
    ("HA10-1L", "2023-04-10", 'V:'), #used in Mabry's thesis
    ("HA10-1L", "2023-04-12", 'V:'),
    ("HA10-1L", "2023-04-17", 'V:'), #difficult dataset reflections/squinty -- NO GLM/HMM
    ("HA11-1R", "2023-04-07", 'V:'), #incosisntent brightness?
    ("HA11-1R", "2023-04-13", 'V:'),
    ("HA11-1R", "2023-05-01", 'V:'),
    ("HA11-1R", "2023-05-02", 'V:'),
    ("HA11-1R", "2023-05-05", 'V:'),
    ("HA2-1L", "2023-04-12", 'V:'), #squinty
    # ("HA2-1L", "2023-04-28", 'V:'), #some nice footage/some occlusion/ERROR message? ends around 47k probably missing frames
    ("HA2-1L", "2023-05-01", 'V:'), #unreliably measurements after 37k
    ("HA2-1L", "2023-05-05", 'V:'),
    # ("HE1-00", "2023-05-30", 'V:'), #-- NO GLM/HMM
    
    # ("HA1-00", "2023-08-28", 'W:'), #overexpossed
    # ("HA1-00", "2023-08-25", 'W:'), #overexpossed
]

# datasets = [("HA2-1L", "2023-05-05", 'V:')]

# Now run your pipeline
pipeline = DataPipeline()

# Load the data from the datasets list
data_loaders, celltype_info, engagement_proj, engagement_test_trials, engagement_frames = pipeline.load_data(datasets,load_celltypes=1) 

# Display the loaded data loaders and cell type info
print("Cell Type Info:", celltype_info)

In [ ]:
#LOAD ALIGNED DATA! TO MAKE PLOTS
from helper_functions.data_pipeline import DataPipeline
pipeline = DataPipeline()
data_loaders, celltype_info = pipeline.load_data(datasets=datasets, load_celltypes=True)

# Initialize processor with alignment parameters
processor = DatasetProcessor(alignment={
    'type': 'pre',
    'data_type': 'deconv'
})

fields_to_separate = ['condition','left_turn'] # ['left_turn'] #is_stim_trial, correct, condition
# 
# Create trial divider for condition splitting
trial_divider = TrialDivider()

# Process and align all datasets
aligned_data = processor.process_datasets(
    data_loaders=data_loaders,
    celltype_info=celltype_info,
    save_path='aligned_data.pkl'
)

# For each dataset, create condition splits
for (animalID, date), data in aligned_data.items():
    print(f"\nProcessing {animalID} {date}")
    
    # Create condition splits
    
    all_conditions, condition_array = trial_divider.divide_trials_from_df(
        trial_info_df=data['trial_info'],
        good_trials=data['good_trials'],
        fields_to_separate=fields_to_separate
    )
    
    # Store conditions in aligned_data for later use
    aligned_data[(animalID, date)]['all_conditions'] = all_conditions
    print(f"Found {len(all_conditions)} conditions")

# Verify alignment worked
for (animalID, date), data in aligned_data.items():
    print(f"\nDataset: {animalID} {date}")
    print(f"Aligned imaging shape: {data['aligned_imaging'].shape}")
    print(f"Number of good trials: {len(data['good_trials'])}")
    print(f"Number of conditions: {len(data['all_conditions'])}")



In [ ]:
#align the data!
